# 01 – Data Exploration
First look at the PHS monthly A&E activity and "When" datasets before cleaning.

In [3]:
import pandas as pd
from pathlib import Path

In [4]:
RAW = Path("../data/raw")

In [10]:
activity_file = next(RAW.glob("monthly_ae_activity*.csv"))
when_file = next(RAW.glob("*when*.csv"))

In [11]:
activity = pd.read_csv(activity_file)
when = pd.read_csv(when_file)

In [12]:
print(activity_file.name, activity.shape)
print(when_file.name, when.shape)

monthly_ae_activity_202608.csv (40066, 26)
opendata_monthly_ae_when_202608.csv (640703, 10)


In [14]:
activity.info()

<class 'pandas.DataFrame'>
RangeIndex: 40066 entries, 0 to 40065
Data columns (total 26 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Month                            40066 non-null  int64  
 1   Country                          40066 non-null  str    
 2   HBT                              40066 non-null  str    
 3   TreatmentLocation                40066 non-null  str    
 4   DepartmentType                   40066 non-null  str    
 5   AttendanceCategory               40066 non-null  str    
 6   NumberOfAttendancesAll           40066 non-null  int64  
 7   NumberWithin4HoursAll            40066 non-null  int64  
 8   NumberOver4HoursAll              40066 non-null  int64  
 9   PercentageWithin4HoursAll        40066 non-null  float64
 10  NumberOfAttendancesEpisode       23076 non-null  float64
 11  NumberOfAttendancesEpisodeQF     16990 non-null  str    
 12  NumberWithin4HoursEpisode    

In [15]:
activity.head()

,Month,Country,HBT,TreatmentLocation,DepartmentType,AttendanceCategory,NumberOfAttendancesAll,NumberWithin4HoursAll,NumberOver4HoursAll,PercentageWithin4HoursAll,...,PercentageWithin4HoursEpisode,PercentageWithin4HoursEpisodeQF,NumberOver8HoursEpisode,NumberOver8HoursEpisodeQF,PercentageOver8HoursEpisode,PercentageOver8HoursEpisodeQF,NumberOver12HoursEpisode,NumberOver12HoursEpisodeQF,PercentageOver12HoursEpisode,PercentageOver12HoursEpisodeQF
0,200707,S92000003,S08000015,A101H,Type 3,Unplanned,252,252,0,100.0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
1,200707,S92000003,S08000015,A101H,Type 3,All,252,252,0,100.0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z
2,200707,S92000003,S08000015,A111H,Type 1,Unplanned,5414,5290,124,97.7,...,97.7,NaN,26.0,NaN,0.5,NaN,24.0,NaN,0.4,NaN
3,200707,S92000003,S08000015,A111H,Type 1,All,5414,5290,124,97.7,...,97.7,NaN,26.0,NaN,0.5,NaN,24.0,NaN,0.4,NaN
4,200707,S92000003,S08000015,A207H,Type 3,Unplanned,92,92,0,100.0,...,NaN,z,NaN,z,NaN,z,NaN,z,NaN,z


### Key observations – activity file
- `Month` is stored as an integer (e.g. 200707) and needs converting to a date.
- Health boards (`HBT`) and hospitals (`TreatmentLocation`) are codes; lookup tables are needed for names.
- `AttendanceCategory` includes "All" alongside "Unplanned"/"New Planned", so filtering is needed to avoid double counting.
- `...All` columns are complete; `...Episode` columns (incl. 8h and 12h waits) only cover sites submitting episode-level data.
- `QF` columns are quality flags to investigate.

In [16]:
when.info()

<class 'pandas.DataFrame'>
RangeIndex: 640703 entries, 0 to 640702
Data columns (total 10 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   Month                640703 non-null  int64
 1   Country              640703 non-null  str  
 2   HBT                  640703 non-null  str  
 3   TreatmentLocation    640703 non-null  str  
 4   DepartmentType       640703 non-null  str  
 5   Day                  640703 non-null  str  
 6   Week                 640703 non-null  str  
 7   Hour                 640703 non-null  str  
 8   InOut                640703 non-null  str  
 9   NumberOfAttendances  640703 non-null  int64
dtypes: int64(2), str(8)
memory usage: 48.9 MB


In [17]:
when.head()

,Month,Country,HBT,TreatmentLocation,DepartmentType,Day,Week,Hour,InOut,NumberOfAttendances
0,201801,S92000003,S08000015,A111H,Type 1,Friday,Weekday,00:00 to 00:59,Out of Hours,20
1,201801,S92000003,S08000015,A111H,Type 1,Friday,Weekday,01:00 to 01:59,Out of Hours,14
2,201801,S92000003,S08000015,A111H,Type 1,Friday,Weekday,02:00 to 02:59,Out of Hours,6
3,201801,S92000003,S08000015,A111H,Type 1,Friday,Weekday,03:00 to 03:59,Out of Hours,5
4,201801,S92000003,S08000015,A111H,Type 1,Friday,Weekday,04:00 to 04:59,Out of Hours,5


### Key observations – "When" file
- No missing values; data starts January 2018.
- Contains attendance counts only (no waiting times), so it shows when demand occurs, not performance by hour.
- Q4 revised to: when do patients arrive, and when is demand highest?
- `Hour` is text (e.g. "00:00 to 00:59") and will need converting for sorting.

In [18]:
for col in activity.select_dtypes(exclude="number").columns :
    print(col, " | ", activity[col].nunique(), activity[col].unique()[:8])

Country  |  1 <StringArray>
['S92000003']
Length: 1, dtype: str
HBT  |  14 <StringArray>
['S08000015', 'S08000016', 'S08000017', 'S08000029', 'S08000019', 'S08000020',
 'S08000031', 'S08000022']
Length: 8, dtype: str
TreatmentLocation  |  103 <StringArray>
['A101H', 'A111H', 'A207H', 'A210H', 'B103H', 'B105H', 'B114H', 'B118H']
Length: 8, dtype: str
DepartmentType  |  2 <StringArray>
['Type 3', 'Type 1']
Length: 2, dtype: str
AttendanceCategory  |  3 <StringArray>
['Unplanned', 'All', 'New planned']
Length: 3, dtype: str
NumberOfAttendancesEpisodeQF  |  1 <StringArray>
['z', nan]
Length: 2, dtype: str
NumberWithin4HoursEpisodeQF  |  1 <StringArray>
['z', nan]
Length: 2, dtype: str
NumberOver4HoursEpisodeQF  |  1 <StringArray>
['z', nan]
Length: 2, dtype: str
PercentageWithin4HoursEpisodeQF  |  1 <StringArray>
['z', nan]
Length: 2, dtype: str
NumberOver8HoursEpisodeQF  |  1 <StringArray>
['z', nan]
Length: 2, dtype: str
PercentageOver8HoursEpisodeQF  |  1 <StringArray>
['z', nan]
Length

### Text column checks – activity file
- `Country` has a single value (Scotland), so it will be dropped.
- 14 health boards (matches Scotland's 14 territorial boards) and 103 treatment locations.
- `DepartmentType`: Type 1 (major EDs) and Type 3 (minor injury units/other); analysed separately.
- `AttendanceCategory` values are "Unplanned", "All" and "New planned" (lowercase p, unlike the documentation).
- `QF` columns contain only "z" (not applicable), marking rows without episode-level data
  (23,076 + 16,990 = 40,066). They add no extra information and will be dropped.

In [19]:
for col in when.select_dtypes(exclude="number").columns:
    print(col, " | ", when[col].nunique(), when[col].unique()[:8])

Country  |  1 <StringArray>
['S92000003']
Length: 1, dtype: str
HBT  |  14 <StringArray>
['S08000015', 'S08000016', 'S08000017', 'S08000019', 'S08000020', 'S08000022',
 'S08000024', 'S08000025']
Length: 8, dtype: str
TreatmentLocation  |  58 <StringArray>
['A111H', 'A210H', 'B120H', 'Y110H', 'Y144H', 'Y146H', 'V201H', 'V217H']
Length: 8, dtype: str
DepartmentType  |  2 <StringArray>
['Type 1', 'Type 3']
Length: 2, dtype: str
Day  |  7 <StringArray>
['Friday', 'Monday', 'Saturday', 'Sunday', 'Thursday', 'Tuesday', 'Wednesday']
Length: 7, dtype: str
Week  |  2 <StringArray>
['Weekday', 'Weekend']
Length: 2, dtype: str
Hour  |  24 <StringArray>
['00:00 to 00:59', '01:00 to 01:59', '02:00 to 02:59', '03:00 to 03:59',
 '04:00 to 04:59', '05:00 to 05:59', '06:00 to 06:59', '07:00 to 07:59']
Length: 8, dtype: str
InOut  |  2 <StringArray>
['Out of Hours', 'In Hours']
Length: 2, dtype: str


### Text column checks – "When" file
- `Country` has a single value and will be dropped.
- Covers 58 treatment locations (vs 103 in the activity file), as only sites submitting
  episode-level data are included. Arrival-pattern findings apply to these sites only.
- `Day`, `Week`, `Hour` and `InOut` contain the expected values with no inconsistencies.
- `Day` sorts alphabetically by default; a Monday–Sunday order must be set before charting.

## Summary
Both datasets load correctly. Cleaning will need to: convert `Month` to dates, filter
`AttendanceCategory` to avoid double counting, drop `Country` and the `QF` columns,
add health board and hospital names, and set correct ordering for days and hours.